# 6장 실습 — RAPTOR 구현과 경로 탐색

작은 시간표에서 환승 경로를 손으로 계산한 뒤 탈 차 찾기와 라운드 갱신을 구현합니다.
계산 결과를 검증하고 하남 자료의 경로 한 건을 읽습니다.

`labs/ch06_raptor.py`에서 채울 함수는 `Pattern.earliest_trip`과 `raptor` 두 개입니다.
`TransitData.from_gtfs`는 제공된 채로 사용합니다. 자료 변환을 직접 작성하는 일은 심화 과제입니다.
지도와 시간대별 분석은 [추가 탐색](extensions/ch06_raptor_exploration.ipynb),
새 노선 추가는 [통합 과제](../projects/gtfs_route_design/route_design.ipynb)로 분리합니다.

먼저 실행 경로와 한글 글꼴을 준비합니다. 아래 셀은 학생 파일 저장 후 자동으로 다시 읽도록 설정합니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import matplotlib.pyplot as plt
from lab import expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

학생 구현은 `sol`, 제공된 탐색은 `ref_raptor`로 읽습니다. 제공된 결과를 볼 수 있어도 학생 함수의 채점과는 구분합니다.

In [ ]:
import ch06_raptor as sol
from smartmob.data import load_gtfs, parse_gtfs_time, seconds_to_gtfs_time
from smartmob.teaching.raptor import (
    INF, TransitData, toy_feed, raptor as ref_raptor, journey, summarize,
)

def clock_label(seconds):
    return "미도달" if seconds == INF else seconds_to_gtfs_time(seconds)

## 1. 시간표 기반 환승 경로 분석 (교재 6.1~6.2)

5장의 1호선은 A 08:00→B 08:10→C 08:20이며 다음 편은 30분 뒤입니다.
2호선 D 08:15→E 08:25를 추가합니다. B에서 D까지 도보는 110초입니다.
종이에 `B 08:10 → 도보 110초 → D ? → 대기 ? → 08:15 탑승 → E ?`를 채웁니다.
답을 적은 뒤 아래 표의 여덟 방문 행과 도보 연결을 확인합니다.

In [ ]:
small = toy_feed()
display(small["stop_times"])
toy = TransitData.from_gtfs(small, max_transfer_m=300)
a, b, c, d, e = [toy.index_of[s] for s in "ABCDE"]
print("B에서의 도보 연결:", [(toy.stop_ids[s], sec) for s, sec in toy.transfers[b]])

D 도착은 08:11:50, 대기는 3분 10초, E 도착은 08:25입니다.
두 번 탔으므로 환승은 한 번입니다. B→D를 걷는 일은 차량 탑승 횟수를 늘리지 않습니다.
A 08:05 출발이라면 B 08:40, D 08:41:50이 되어 유일한 2호선 운행을 놓칩니다.

### 고정 비용 그래프와 시간표 모형의 결과 비교

3장에서 사용한 다익스트라에 차내시간과 도보시간만 넣으면 어떤 값이 나올지 예상합니다.
A→B 600초, B→C 600초, B↔D 110초, D→E 600초로 둡니다. 이 그래프에는 차량 출발시각이 없습니다.
`adj`의 각 항목은 3장과 같은 `(이웃, 소요시간, 엣지 번호)`입니다.
좌표와 속도는 이번 비용 계산에 쓰지 않으므로 비워 두고 인접 리스트만 입력합니다.
제공된 다익스트라와 RAPTOR를 실행해 A 08:00·08:05 출발의 E 도착을 비교합니다.

In [ ]:
from smartmob.teaching.graph import RoadGraph
from smartmob.teaching.dijkstra import dijkstra

fixed_graph = RoadGraph(
    adj={
        "A": [("B", 600, 0)],
        "B": [("C", 600, 1), ("D", 110, 2)],
        "C": [],
        "D": [("B", 110, 3), ("E", 600, 4)],
        "E": [],
    },
    coord={}, edges=pd.DataFrame(), speed_column="",
)
fixed_path = dijkstra(fixed_graph, "A", "E")
print("고정 비용 경로:", " → ".join(fixed_path.nodes))
print("소요시간(초):", fixed_path.duration_s)
comparison = []
for departure_text in ["08:00:00", "08:05:00"]:
    departure = parse_gtfs_time(departure_text)
    scheduled = ref_raptor(toy, [(a, 0)], departure, max_rounds=2)
    comparison.append({
        "A 출발": departure_text,
        "시간표 없는 그래프 + 다익스트라": clock_label(departure + fixed_path.duration_s),
        "시간표 + RAPTOR": clock_label(scheduled.best[e]),
    })
pd.DataFrame(comparison)


고정 비용 경로는 A→B→D→E, 소요시간은 1,310초(21분 50초)입니다.
08:00 출발에서는 E 08:21:50으로 계산하지만 시간표 결과는 08:25입니다. D의 대기 3분 10초가 빠졌습니다.
08:05 출발에서도 그래프는 E 08:26:50으로 계산하지만, 실제로는 유일한 D 08:15 차를 놓쳐 미도달입니다.
다익스트라는 주어진 그래프의 최단경로를 구했습니다. 차이는 그 그래프에 시간표가 없기 때문에 생깁니다.
교재 6.1절처럼 시각별 노드와 대기 엣지를 넣으면 다익스트라도 올바른 시간표 경로를 찾을 수 있습니다.

### 탑승 횟수별 도착시각

아래 셀을 실행하기 전에 라운드 0·1·2에서 C와 E에 도달하는지 각각 적습니다.
라운드 k는 k번 이하로 탄 결과이며 이전에 얻은 빠른 답도 남깁니다.

In [ ]:
toy_result = ref_raptor(toy, [(a, 0)], 8 * 3600)
round_table = pd.DataFrame(toy_result.rounds, columns=toy.stop_ids)
round_table.index.name = "라운드"
round_table.map(clock_label)

라운드 1은 C 08:20·D 08:11:50, E 미도달입니다. 라운드 2에서 E 08:25가 생깁니다.
C는 이미 한 번 타서 빠르게 도착하므로 라운드 2에서도 08:20입니다.
[계산 과정 화면](../_static/ch06_raptor.html)에서 같은 시간표를 한 단계씩 확인할 수 있습니다.

## 2. 패턴·정류장 색인·도보 연결의 구조 (교재 6.3)

패턴 0은 1호선 A→B→C, 패턴 1은 2호선 D→E입니다.
정류장 전체 번호는 A=0, B=1, C=2, D=3, E=4입니다.
D는 전체 번호 3이지만 패턴 1 안에서는 첫 정류장이므로 위치 0입니다.
`routes_by_stop[D]`의 `(1, 0)`과 B의 도보 연결 `(3, 110)`을 먼저 말로 풀어 봅니다.

In [ ]:
mine = sol.TransitData.from_gtfs(small, max_transfer_m=300)
ma, mb, mc, md_, me = [mine.index_of[s] for s in "ABCDE"]
print("패턴 수:", len(mine.patterns))
print("D에서 읽을 패턴·위치:", mine.routes_by_stop[md_])
print("B에서 걸어갈 정류장·초:", mine.transfers[mb])
for pattern in mine.patterns:
    print(pattern.name, [mine.stop_ids[s] for s in pattern.stops])

패턴은 2개입니다. D에서는 패턴 1의 위치 0부터 읽고, B에서는 D까지 110초 걸을 수 있습니다.
표를 이렇게 준비하는 함수는 이미 제공되어 있습니다. 다음으로 시각 배열의 행과 열을 읽습니다.

In [ ]:
pattern = mine.patterns[0]
departure_table = pd.DataFrame(pattern.departures, columns=["A", "B", "C"])
display(departure_table.map(clock_label))
print("운행 1, 위치 0:", clock_label(pattern.departures[1][0]))
print("운행 0, 위치 1:", clock_label(pattern.departures[0][1]))

행은 운행 번호, 열은 그 패턴 안의 정류장 위치입니다. `[1][0]`은 두 번째 차의 A 08:30입니다.
A에서 탈 차는 A 열, B에서 탈 차는 B 열에서 고릅니다.

## 3. 탑승 가능 운행 탐색의 구현 (교재 6.3)

입력은 정류장 위치와 준비 시각, 출력은 운행 번호입니다. 운행 0과 탈 차 없음 `None`을 구분합니다.
A 열 `[08:00, 08:30]`에서 07:55·08:00·08:03·08:30:01의 답을 먼저 적습니다.
순서대로 0·0·1·없음입니다. `bisect_left`의 반환 위치가 목록 길이와 같으면 탈 차가 없습니다.

In [ ]:
from bisect import bisect_left

departures = [parse_gtfs_time(t) for t in ["08:00:00", "08:30:00"]]
for text in ["07:55:00", "08:00:00", "08:03:00", "08:30:01"]:
    i = bisect_left(departures, parse_gtfs_time(text))
    print(text, "→", clock_label(departures[i]) if i < len(departures) else "탈 차 없음")

정각은 첫 차를 탈 수 있고 08:03은 다음 차를 고릅니다. 마지막 출발을 지나면 반환 위치가 2입니다.
이 규칙으로 `sol.Pattern.earliest_trip`을 채운 뒤 세 조건을 검사합니다.

In [ ]:
line1 = sol.Pattern(
    name="1호선", route_type=1, stops=[0, 1, 2],
    arrivals=[[28800, 29400, 30000], [30600, 31200, 31800]],
    departures=[[28800, 29400, 30000], [30600, 31200, 31800]],
)
line1.build_index()
try:
    assert line1.earliest_trip(0, 28800) == 0
    assert line1.earliest_trip(0, 28980) == 1
    assert line1.earliest_trip(0, 30601) is None
    print("정각 탑승, 다음 차, 마지막 차 이후를 확인했습니다.")
except NotImplementedError:
    print("earliest_trip을 채운 뒤 다시 실행합니다.")

세 조건을 통과하면 B 위치에서도 08:12에 운행 1을 고르는지 확인합니다.
순서가 뒤집힌 목록 `[08:30, 08:20]`에는 이분 탐색을 그대로 쓸 수 없습니다.
파일의 안내대로 `_sorted`가 거짓이면 목록을 훑어 준비 시각 이상인 출발 중 최솟값을 찾습니다.

In [ ]:
unsorted = sol.Pattern(
    name="순서 확인", route_type=1, stops=[0, 1],
    arrivals=[[28800, 30600], [29400, 30000]],
    departures=[[28800, 30600], [29400, 30000]],
)
unsorted.build_index()
try:
    assert unsorted.earliest_trip(1, 29700) == 1
    print("위치 1에서 08:20에 출발하는 1번 운행을 선택했습니다.")
except NotImplementedError:
    print("정렬되지 않은 출발시각 처리도 구현합니다.")

08:15에 준비된 승객은 08:20에 출발하는 운행 1을 고릅니다. 이 검사는 출발편 선택 규칙을 확인합니다.
뒤 차가 추월하는 전체 경로 문제의 한계는 교재 6.6절에서 읽습니다.

A의 08:03과 B의 08:12에 탈 차를 시각 배열에서 짚어 설명합니다.

## 4. 첫 번째 라운드의 상태 갱신 (교재 6.4)

종이에 이전 답과 현재 답을 각각 다섯 칸으로 그립니다. 처음에는 A만 08:00이며 나머지는 미도달입니다.
A의 이전 답으로 첫 차를 선택합니다. 같은 차의 B 08:10, C 08:20을 현재 답에 씁니다.
B에서 110초를 더한 D 08:11:50도 현재 답입니다. E는 아직 미도달입니다.

D의 새 값을 즉시 승차에 쓰면 한 라운드에 두 번 타게 됩니다.
아래 셀은 손으로 구한 라운드 1을 적는 연습입니다. B·C의 시각은 선택한 첫 차의 표에서 읽습니다.
3장의 `prev`는 이전 노드였지만 여기서는 이전 라운드의 도착시각입니다.
3장의 `done`처럼 한 번 처리한 정류장을 제외하면 다음 라운드의 개선을 놓칠 수 있습니다.


In [ ]:
prev = [8 * 3600, INF, INF, INF, INF]
cur = list(prev)
cur[mb] = mine.patterns[0].arrivals[0][1]
cur[mc] = mine.patterns[0].arrivals[0][2]
cur[md_] = cur[mb] + 110
pd.DataFrame({"이전 답": prev, "현재 답": cur}, index=mine.stop_ids).map(clock_label)

현재 답에 B·C·D가 생겨도 이전 답은 그대로입니다. E는 미도달이고 C는 08:20입니다.
이 현재 답을 다음 이전 답으로 넘긴 뒤에야 D의 08:11:50으로 08:15 차를 탈 수 있습니다.

## 5. RAPTOR 라운드 갱신의 구현 (교재 6.4~6.5)

`sol.raptor`를 초기화 → 패턴 선택 → 승하차 → 도보 → 종료의 순서로 작성합니다.
승차에는 `prev`, 새 도착 기록에는 `cur`를 씁니다. B의 이전 답이 미도달이어도 A에서 탄 차는 계속 따라갑니다.
먼저 최대 탑승 횟수 0·1·2에서의 결과를 아래 예상과 비교합니다.

- 0회: A만 도달합니다.
- 1회: C 08:20, D 08:11:50이며 E는 미도달입니다.
- 2회: E 08:25가 생기고 C의 08:20은 유지됩니다.

In [ ]:
try:
    mine = sol.TransitData.from_gtfs(small, max_transfer_m=300)
    ma, mc, me = [mine.index_of[s] for s in ["A", "C", "E"]]
    rows = [sol.raptor(mine, [(ma, 0)], 28800, max_rounds=k) for k in range(4)]
    assert rows[0][mc] == INF
    assert rows[1][mc] == 30000 and rows[1][me] == INF
    assert rows[2][me] == 30300
    display(pd.DataFrame(rows, columns=mine.stop_ids).map(clock_label))
except NotImplementedError:
    print("raptor를 채운 뒤 다시 실행합니다.")

E가 라운드 1에 나오면 승차 판단에 현재 답을 썼는지 확인합니다.
다음 검사를 실행하기 전에 세 경우를 손으로 계산합니다.
A 08:05는 첫 차를 놓치고, 07:55에 10분을 걸어도 A 08:05 도착입니다. 23:00에는 남은 운행이 없습니다.

In [ ]:
try:
    late = sol.raptor(mine, [(ma, 0)], 29100)
    access = sol.raptor(mine, [(ma, 600)], 28500)
    night = sol.raptor(mine, [(ma, 0)], 23 * 3600)
    assert late[mc] == 31800 and late[me] == INF
    assert access[mc] == 31800
    assert night[mc] == INF and night[me] == INF
    print("놓친 첫 차, 접근 도보, 마지막 운행 이후를 확인했습니다.")
except (NotImplementedError, NameError):
    print("학생 구현을 완성한 뒤 다시 실행합니다.")

앞의 두 경우는 C 08:50이며 E는 미도달입니다. 23:00에는 C도 미도달입니다.
값이 다르면 시각 단위, 정각 승차 조건, 접근 도보의 합산 순서부터 확인합니다.

## 6. 구현 결과의 검증 (교재 6.5)

두 함수를 저장하고 채점합니다. 준비 함수는 제공된 채로 사용해도 됩니다.
채점은 작은 시간표의 조건과 실제 하남 자료의 불변식을 확인합니다.

In [ ]:
from check import check

report = check("ch06")

아직 빈칸이면 자료 준비 검사는 통과해도 탐색 검사는 실패합니다. 탐색까지 모두 통과해야 기본 구현이 완료됩니다.
제출물은 `.py` 파일, 채점 출력, 작은 시간표에서 확인한 조건 세 가지의 설명입니다.

## 7. 하남 대중교통 경로의 해석 (교재 6.6)

작은 예제의 답을 설명한 뒤 실제 자료로 옮깁니다. 제공된 함수로 경로 한 건을 읽습니다.
표를 준비하는 시간과 경로를 찾는 시간은 별도로 측정합니다. 여기서는 새로운 함수를 구현하지 않습니다.

In [ ]:
from time import perf_counter

hanam = load_gtfs("hanam")
started = perf_counter()
data = TransitData.from_gtfs(hanam)
print("준비 시간(초):", round(perf_counter() - started, 2))
print(data.describe())
print("출발순이 뒤집힌 패턴:", sum(not p._sorted for p in data.patterns))

패턴 349개, 운행 8,923개, 방향을 구분한 도보 연결 28,818개입니다.
같은 패턴의 운행이 추월하는 경우 등 제공 구현의 간소화는 교재 6.6절에 있습니다.
출발지는 하남시청 건물 좌표입니다. 정류장까지 걷는 시간을 더한 뒤 어느 차를 탈지 찾습니다.

In [ ]:
origin = (37.5393, 127.2148)
destination = (37.5606, 127.1930)
origins = data.access_stops(*origin)
target = data.nearest_stop(*destination)
departure = 8 * 3600
started = perf_counter()
result = ref_raptor(data, origins, departure)
query_ms = (perf_counter() - started) * 1000
print("접근 정류장:", len(origins), "도착 정류장:", data.stop_names[target])
print("유한한 도착시각:", sum(t < INF for t in result.best), "개")
print("질의 시간(ms):", round(query_ms, 1))

선택된 목적지 정류장은 ‘미사강변브라운스톤’입니다. 시간표와 최대 다섯 번 탑승으로 4,156곳에 도착시각이 나옵니다.
이제 목적지 한 곳의 경로를 읽습니다. 각 구간에서 언제 타고 내렸는지, 다음 승차까지 얼마나 기다리는지 확인합니다.

In [ ]:
legs = journey(data, result, target)
for leg in legs:
    if leg["kind"] == "transit":
        print(leg["route"], data.stop_names[leg["from"]], "→", data.stop_names[leg["to"]],
              clock_label(leg["board_time"]), "→", clock_label(leg["alight_time"]))
    else:
        print("도보", leg["seconds"], "초 →", data.stop_names[leg["to"]])
summary = summarize(data, legs, departure)
summary

약 22.9분은 차내 6.6분·도보 11.2분·대기 5.1분의 합계입니다.
출발지 접근 도보는 포함하고, 마지막 정류장에서 목적지 건물까지의 도보는 포함하지 않습니다.

[추가 탐색](extensions/ch06_raptor_exploration.ipynb)에서는 출발시각·탑승 상한·도달 범위를 바꿉니다.
[통합 과제](../projects/gtfs_route_design/route_design.ipynb)에서는 신규 노선을 만들고 추가 전후를 비교합니다.
`TransitData.from_gtfs` 직접 작성은 교재 6.7절의 심화 과제입니다.